<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andres22feli/si7011-AgudeloOrtega-GarnicaBohorquez/blob/main/evento3/sesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

2026/11/13 21:15:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/13 21:15:54 INFO mlflow.store.db.utils: Updating database tables
2026/11/13 21:15:56 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.22.1 · trl 1.16.0 · peft 0.21.2 · MLflow 3.18.0 · cuda


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [3]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

100%|██████████| 93.9k/93.9k [00:00<00:00, 54.9MB/s]

Extracting files...
{'fuente': '/root/.cache/kagglehub/datasets/juanmartinezv4399/si7011-s06-dolly-quijote/versions/2/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


In [4]:
df.shape, train.shape, val.shape, test.shape

((384, 6), (284, 6), (40, 6), (59, 6))

In [5]:
df.head()

,id,category,instruction,response,response_estilo,split
0,41,open_qa,¿Cuándo se inventó la rueda?,Nadie sabe cuándo se fabricó la rueda por prim...,Nadie puede decir con certidumbre cuándo se fa...,train
1,2863,general_qa,Háblame del descubrimiento del grano de café.,El café fue descubierto por primera vez por el...,En verdad os digo que el grano de café fue des...,train
2,3301,general_qa,¿Quiénes son los Cardenales de San Luis?,Los Cardenales de San Luis son un equipo de bé...,"Aqueste es el Cardenal de San Luis, un prestig...",train
3,9937,general_qa,¿Cuántas lenguas se hablan en Nigeria?,En Nigeria se hablan más de 500 lenguas nativa...,Aqueste espléndido país de Nigeria veo yo que ...,train
4,10420,brainstorming,¿Cómo hacer una masa de Pizza?,Mezcla aproximadamente 1 taza (125 g) de harin...,"Mezclad, os digo, vuestra merced, unas doce on...",train


In [7]:
val.head()

,id,category,instruction,response,response_estilo,split
0,41,open_qa,¿Cuándo se inventó la rueda?,Nadie sabe cuándo se fabricó la rueda por prim...,Nadie puede decir con certidumbre cuándo se fa...,train
1,2863,general_qa,Háblame del descubrimiento del grano de café.,El café fue descubierto por primera vez por el...,En verdad os digo que el grano de café fue des...,train
2,3301,general_qa,¿Quiénes son los Cardenales de San Luis?,Los Cardenales de San Luis son un equipo de bé...,"Aqueste es el Cardenal de San Luis, un prestig...",train
3,9937,general_qa,¿Cuántas lenguas se hablan en Nigeria?,En Nigeria se hablan más de 500 lenguas nativa...,Aqueste espléndido país de Nigeria veo yo que ...,train
4,10420,brainstorming,¿Cómo hacer una masa de Pizza?,Mezcla aproximadamente 1 taza (125 g) de harin...,"Mezclad, os digo, vuestra merced, unas doce on...",train


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [8]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?
  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]
  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.
  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la 

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [10]:
def a_conversacion(fila, sistema=None):
    # Convierte una fila del dataset en una conversación para entrenamiento.
    return {
        # PROMPT: contiene la pregunta del usuario con opción de
        # una instrucción de sistema que indica cómo debe responder.
        'prompt': mensajes(fila['instruction'], sistema),

        # contiene la respuesta que queremos que aprenda el modelo.
        'completion': [
            {
                # identifica que el texto corresponde a la respuesta del asistente.
                'role': 'assistant',

                # extrae la respuesta escrita con el estilo de Don Quijote.
                'content': fila['response_estilo']
            }
        ]
    }



In [11]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?', 'role': 'user'}], 'completion': [{'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.', 'role': 'assistant'}]}


La prueba confirma que se prepararon 284 conversaciones de entrenamiento, cada una con su pregunta (prompt) y su respuesta en el estilo de Don Quijote (completion).

## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

In [13]:

# Calcula la longitud real de cada conversación en tokens
longitudes = []

for fila in D_train:
    # Une la pregunta y la respuesta
    conversacion = fila["prompt"] + fila["completion"]

    # Convierte la conversación en una secuencia de tokens
    tokens = tok.apply_chat_template(
        conversacion,
        tokenize=True,
        return_dict=False
    )

    # Cuenta los tokens generados
    longitudes.append(len(tokens))

# Muestra un resumen de las longitudes
print("Longitud mínima:", min(longitudes))
print("Longitud promedio:", sum(longitudes) / len(longitudes))
print("Longitud máxima:", max(longitudes))

# Identifica las conversaciones que superarían el límite de 384 tokens
print("Conversaciones con más de 384 tokens:",
      sum(n > 384 for n in longitudes))



Longitud mínima: 59
Longitud promedio: 157.80633802816902
Longitud máxima: 363
Conversaciones con más de 384 tokens: 0


In [16]:
# TODO 2: Configuración de LoRA y del entrenamiento SFT

# LoRA permite adaptar el modelo entrenando matrices pequeñas,
# sin modificar directamente todos los pesos originales.
lora = LoraConfig(
    r=16,                          # Rango de las matrices LoRA
    lora_alpha=32,                 # Factor de escala de la adaptación
    lora_dropout=0.0,              # Sin dropout en LoRA
    task_type='CAUSAL_LM',         # Modelo que predice el siguiente token
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj'
    ]
)

# SFTConfig establece las condiciones del entrenamiento supervisado.
args = SFTConfig(
    output_dir='sft_estilo',       # Carpeta de resultados
    run_name='1-lora-r16',         # Nombre del experimento

    completion_only_loss=True,     # Calcula la pérdida solo sobre la respuesta
    max_length=384,                # Límite validado: máximo observado

    num_train_epochs=2,           # Dos recorridos del dataset
    per_device_train_batch_size=8, # Ocho ejemplos por lote
    per_device_eval_batch_size=8,  # Ocho ejemplos por lote de validación

    learning_rate=5e-4,           # Tasa de aprendizaje
    lr_scheduler_type='cosine',    # Reduce gradualmente la tasa

    fp16=(dev == 'cuda'),         # Precisión de 16 bits cuando hay GPU
    bf16=False,                   # Desactivado para GPU T4

    eval_strategy='steps',        # Evalúa durante el entrenamiento
    eval_steps=10,                # Validación cada 10 pasos
    logging_steps=5,              # Registra métricas cada 5 pasos
    save_strategy='no',           # No guarda checkpoints intermedios
    report_to='mlflow'            # Registra los resultados en MLflow
)

Resumen de la configuración:

* loRA: rango r= 16, aplicado a las proyecciones Q,K, V, O.
* SFT: dos épocas de entrenamiento, lotes de 8 ejemplos y tasa de aprendizaje de 5x10^-4
* Longitud máxima: 384 tokens, suficiente para las 284 conversaciones analizadas.
* Pérdida: claculada sobre los tokens de la respuesta.  



In [17]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el gran diferencial entre Spark y MapReduce: mientras que MapReduce opera y guarda los datos en disco para cada paso, hube de decir que Spark los conserva en memoria para los pasos subsiguientes, lo que le confiere una notable agilidad y rapidez.<|im_end|>\n'


Significa que estamos adaptando un modelo de aproximadamente 496 millones de parámetros, pero solo entrenaremos 2,16 millones (LoRA con el 0,4359 % de sus parámetros)

## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [18]:
t0 = time.time()
trainer.train()
print(f'{time.time() - t0:.0f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.225165,2.253568,2.325377,0.539545,12893.000000
20,2.147027,2.154488,2.228458,0.553987,25677.000000
30,2.177206,2.125636,2.212306,0.562978,37567.000000
40,1.873731,2.109611,2.103669,0.561617,49544.000000
50,1.880957,2.107389,2.006092,0.559956,62739.000000
60,1.873208,2.098426,2.022703,0.560106,74944.000000
70,1.860720,2.096363,2.032239,0.560052,87241.000000
72,1.860720,2.096261,2.032313,0.560241,89634.000000


51 s
adaptador y contrato en MLflow, corrida 407f3e0b629447618de51a217b85bf82


El entrenamiento con LoRA terminó correctamente: 72 pasos, 2 épocas y 89.634 tokens procesados.

In [19]:
def nll(datos):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [20]:
def comparar(filas):
    # Prepara las preguntas y sus respuestas objetivo,
    # sin añadir instrucciones especiales de estilo.
    datos = conjunto(filas)

    # Evalúa el modelo original, con LoRA desactivado.
    with trainer.model.disable_adapter():
        J_base = nll(datos)

        # Evalúa el modelo original, pero ahora con una
        # instrucción de sistema que solicita el estilo literario.
        datos_estilo = conjunto(filas, ESTILO)
        J_prompt = nll(datos_estilo)

    # Fuera del bloque anterior, LoRA vuelve a estar activo.
    # Evalúa el modelo adaptado, sin instrucción de estilo.
    J_lora = nll(datos)

    # Devuelve las tres pérdidas para compararlas.
    return {
        'base': J_base,
        'base + estilo en el prompt': J_prompt,
        'LoRA': J_lora
    }


In [21]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (63 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.944403,0,1.836912,0.597582,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.944        6.989


Resultados del Conjunto de prueba:

* Modelo base: 2.593 (pérdida J) / 13.368 (perplejidad)
* Base + promt de estilo: 2.506 (pérdida J) / 12.261 (perplejidad)
* Modelo con LoRA: 1.944 (pérdida J) / 6.989 (perplejidad)

Adaptar el modelo con LoRA produce una mejora mucho mayor: la pérdida disminuye hasta 1.944, con una perplejidad de 6.989.

Aunque LoRA obtiene menor pérdida y perplejidad; necesitamos leer respuestas generadas para comprobar si realmente conserva el sentido de la pregunta, utiliza el estilo literario y evita expresiones artificiales o repetitivas.

$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [22]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                              0.00                    76.2
base + estilo en el prompt   2.506       12.261                              0.00                    55.9
                      LoRA   1.944        6.989                              0.85                    56.6 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

LoRA consigue una pérdida mucho menor y reproduce las expresiones del conjunto de entrenamiento. Sin embargo, algunas respuestas pierden precisión factual, coherencia o utilidad.

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_

**Nuestro cambio:**  reducir el rango de LoRA de \(r=16\) a \(r=4\), manteniendo los demás hiperparámetros.

**Nuestra predicción:** al reducir el rango, disminuirá el número de parámetros entrenables y posiblemente aumentará ligeramente la pérdida de prueba, porque el modelo tendrá menor capacidad de adaptación. Aunque, podría conservar gran parte del estilo aprendido.

In [23]:

# SECCIÓN 7: Segundo experimento con LoRA r=4

# Cargar nuevamente el modelo original, sin el adaptador entrenado.
modelo_2 = AutoModelForCausalLM.from_pretrained(
    MODELO, dtype=torch.float32
).to(dev)

# Configurar LoRA con rango 4 en lugar de 16.
lora_2 = LoraConfig(
    r=4,                          # ÚNICO cambio experimental: rango 16 -> 4
    lora_alpha=32,                # Mantiene el valor anterior
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj'
    ]
)

# Conservar la configuración del primer entrenamiento.
args_2 = SFTConfig(
    output_dir='sft_estilo_r4',   # Carpeta diferente para no sobrescribir
    run_name='2-lora-r4',        # Identificar el segundo experimento

    completion_only_loss=True,
    max_length=384,

    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-4,
    lr_scheduler_type='cosine',

    fp16=(dev == 'cuda'),
    bf16=False,

    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='mlflow'
)

# Crear un entrenador independiente para el segundo experimento.
trainer_2 = SFTTrainer(
    model=modelo_2,
    args=args_2,
    train_dataset=D_train,
    eval_dataset=D_val,
    processing_class=tok,
    peft_config=lora_2
)

# Muestra cuántos parámetros se entrenarán.
trainer_2.model.print_trainable_parameters()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [24]:
# Entrena el segundo modelo con LoRA r=4.
# Se mantienen los mismos datos y la misma configuración de entrenamiento.

resultado_2 = trainer_2.train()

# Muestra las métricas finales del entrenamiento.
print("Entrenamiento LoRA r=4 completado")
print(resultado_2.metrics)


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.260476,2.283704,2.402583,0.532935,12893.000000
20,2.161023,2.158359,2.226247,0.552537,25677.000000
30,2.176722,2.127672,2.222363,0.560261,37567.000000
40,1.901431,2.109954,2.111111,0.559631,49544.000000
50,1.917100,2.105881,2.034458,0.559993,62739.000000
60,1.884761,2.097939,2.057233,0.561962,74944.000000
70,1.882743,2.096439,2.066144,0.563071,87241.000000
72,1.882743,2.096368,2.066131,0.563201,89634.000000


Entrenamiento LoRA r=4 completado
{'train_runtime': 47.3133, 'train_samples_per_second': 12.005, 'train_steps_per_second': 1.522, 'total_flos': 303634927681536.0, 'train_loss': 2.0431551535924277, 'epoch': 2.0}


In [25]:
# Evalúa LoRA r=4 sobre las mismas 59 conversaciones de prueba.
# Utilizamos el trainer_2 para no modificar el modelo r=16.

datos_test = conjunto(test)

# Calcula la pérdida del nuevo adaptador
resultado_test_r4 = trainer_2.evaluate(
    eval_dataset=datos_test
)

J_r4 = resultado_test_r4["eval_loss"]

# Convierte la pérdida en perplejidad
ppl_r4 = np.exp(J_r4)

# Compara con los resultados previos de LoRA r=16
print("Comparación en el conjunto de prueba")
print(f"LoRA r=16: J = {1.944403:.6f}, PPL = {np.exp(1.944403):.3f}")
print(f"LoRA r=4 : J = {J_r4:.6f}, PPL = {ppl_r4:.3f}")


Tokenizing eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
1.882743,1.951111,72,1.867279,0.596031,89634.000000


Comparación en el conjunto de prueba
LoRA r=16: J = 1.944403, PPL = 6.989
LoRA r=4 : J = 1.951111, PPL = 7.036


Resultados:

Los parámetros entrenables disminuyeron de 2.162.688 a 540.672 (75 % menos). La pérdida de prueba aumentó ligeramente de 1.944403 a 1.951111 y la perplejidad de 6.989 a 7.036.

## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [26]:

# Guarda modelos cargados para reutilizarlos en futuras consultas.
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):

    # 1. Lee el contrato guardado junto con el adaptador LoRA.
    ruta = Path(carpeta)
    with open(ruta / 'contrato.json', 'r', encoding='utf-8') as f:
        contrato = json.load(f)

    # 2. Comprueba que la pregunta sea texto y no esté vacía.
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser un texto no vacío.')

    # 3. Cuenta los tokens de la pregunta.
    n_tokens = len(tok.encode(pregunta, add_special_tokens=False))

    # Rechaza preguntas que excedan el límite del contrato.
    if n_tokens > contrato['max_tokens_pregunta']:
        raise ValueError(
            f'Pregunta demasiado larga: {n_tokens} tokens; '
            f'máximo {contrato["max_tokens_pregunta"]}.'
        )

    # 4. Comprueba si el modelo ya fue cargado anteriormente.
    clave = str(ruta.resolve())

    if clave not in _CACHE:

        # Carga el modelo base indicado en el contrato.
        base = AutoModelForCausalLM.from_pretrained(
            contrato['modelo'],
            dtype=torch.float32
        ).to(dev)

        # Incorpora el adaptador LoRA almacenado en la carpeta.
        modelo_ajustado = PeftModel.from_pretrained(
            base, str(ruta)
        )

        # Activa el modo evaluación, sin actualizar parámetros.
        modelo_ajustado.eval()

        # Guarda el modelo para evitar cargarlo nuevamente.
        _CACHE[clave] = modelo_ajustado

    # 5. Recupera el modelo con LoRA activo.
    modelo_ajustado = _CACHE[clave]

    # 6. Genera la respuesta sin instrucción de sistema.
    return responder(
        modelo_ajustado,
        pregunta,
        max_nuevos=contrato['max_nuevos']
    )


In [27]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

1. Consistencia ✓ · La bicicleta de gravilla, que se llama así porque su principal función es llevarse en el aire, es una máquina que se utiliza para moverse por [...]
1. Consistencia ✓ · Aqueste es el chiste que os digo: Hube de hacer una visita a la Antártica y me di cuenta de que no hay nada más original que las aves [...]
1. Consistencia ✓ · Aqueste es el hallazgo que os digo: la mejor película de todos los tiempos es "La Tierra en el Cielo" de James Cameron, que se estrenó en 1997 y [...]
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → Pregunta demasiado larga: 402 tokens; máximo 128.


El despliegue funciona técnicamente, pero eso no garantiza que las respuestas sean factualmente correctas. Las respuestas sobre la bicicleta de gravilla y la película siguen mostrando los errores que identificamos en TODO 3.

## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?